# Day 01 — Type Hints & Pydantic v2 Basics

**Phase 1 · Module 1: LangGraph Core** · ~30 min

### What I practice today
- [ ] Add **type hints** to every function I write
- [ ] Build two Pydantic models: `AgentInput`, `AgentOutput`
- [ ] Use `model_validate()` and `model_dump()` on agent (LangChain chain) I/O

### Why this matters
LangGraph agents pass data between nodes. If a node gets the wrong shape of data, the agent breaks **silently** — deep inside, hard to debug. Type hints + Pydantic catch the bad data **at the door**, with a clear error.

> Rule of thumb: *type hints document intent, Pydantic enforces it at runtime.*


## 1. Type Hints — what they are

A type hint says *what kind of value* goes in and comes out of a function. Python does **not** enforce them at runtime (they are hints), but your editor, `mypy`, and humans all read them.

No hints — what is `x`? what comes back? You guess:


In [30]:
# No hints: unclear, easy to misuse
def add_tax(amount, rate):
    return amount + amount * rate

add_tax(100, 0.2)


120.0

Same function **with** hints — intent is obvious:

In [31]:
# With hints: float in, float out
def add_tax(amount: float, rate: float) -> float:
    return amount + amount * rate

add_tax(100.0, 0.2)


120.0

### Common hint building blocks

| Hint | Means |
|------|-------|
| `int`, `str`, `float`, `bool` | basic types |
| `list[str]` | list of strings |
| `dict[str, int]` | dict, string keys, int values |
| `Optional[str]` | a string **or** `None` |
| `Literal["a", "b"]` | only these exact values allowed |
| `-> None` | function returns nothing |


In [32]:
from typing import Optional, Literal

# list[str]  -> a list where every item is a string
def join_words(words: list[str]) -> str:
    return " ".join(words)

# dict[str, int] -> token counts per model
def total_tokens(usage: dict[str, int]) -> int:
    return sum(usage.values())

print(join_words(["hello", "agent"]))
print(total_tokens({"prompt": 12, "completion": 30}))


hello agent
42


### Why `Optional` — the "maybe missing" case

An agent run may **fail** and produce no answer. The result field is *either a string or nothing*. That is exactly `Optional[str]` (same as `str | None`).

Without it you'd be tempted to return `""` for "no answer" — but then you can't tell *"empty answer"* apart from *"failed, no answer"*. `Optional` makes "missing" a real, distinct value.


In [33]:
from typing import Optional

# error is None on success, a message on failure
def run_agent(query: str) -> Optional[str]:
    if not query.strip():
        return None          # nothing to answer -> explicitly None
    return f"Answer to: {query}"

print(run_agent("what is RAG?"))   # -> str
print(run_agent("   "))            # -> None (clearly "no result")


Answer to: what is RAG?
None


### Why `Literal` — lock a field to a fixed set

An agent's status is one of a **known, closed set**: `success`, `error`, `pending`. If you type it as `str`, then `"succes"` (typo) or `"banana"` are "valid" strings — bug ships silently.

`Literal["success", "error", "pending"]` says *only these three values are allowed*.

⚠️ **Important — a plain `Literal` hint is NOT enforced at runtime.** Like all type hints, `Literal` is checked by your **editor / mypy only**. Python itself ignores it, so `make_status("done")` still runs and returns `status=done`. The hint catches the typo *while you write code*, not while it runs.

To make `Literal` actually **reject** bad values at runtime, put it on a Pydantic model (shown next). That is a core reason Pydantic exists.


In [34]:
from typing import Literal

Status = Literal["success", "error", "pending"]

def make_status(s: Status) -> str:
    return f"status={s}"

print(make_status("success"))

# Runtime does NOT enforce Literal -> this still runs! (editor/mypy would flag it)
print(make_status("done"))   # status=done  <- bad value slips through at runtime


status=success
status=done


In [35]:
from typing import Literal
from pydantic import BaseModel, ValidationError

# Same Literal, but now ON a Pydantic model -> enforced at RUNTIME
class Result(BaseModel):
    status: Literal["success", "error", "pending"]

print(Result(status="success"))      # ok

try:
    Result(status="done")            # rejected at runtime, unlike make_status()
except ValidationError as e:
    print(e)


status='success'
1 validation error for Result
status
  Input should be 'success', 'error' or 'pending' [type=literal_error, input_value='done', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 2. Pydantic — hints that actually enforce at runtime

Problem: type hints are **not checked** when the code runs. This passes happily and blows up later:


In [36]:
def plain_agent_input(query: str, max_tokens: int):
    return query, max_tokens

# hint says int, but a string sneaks in -> NO error here
bad = plain_agent_input("hi", "lots")
print(bad)   # ('hi', 'lots')  -> garbage rides along


('hi', 'lots')


**Pydantic** turns a class of hints into a runtime gate. Subclass `BaseModel`, declare fields with hints — Pydantic validates and *coerces* on creation, and raises a clear error on bad data.

In [37]:
from pydantic import BaseModel

class AgentInput(BaseModel):
    query: str
    max_tokens: int

# valid
ok = AgentInput(query="what is RAG?", max_tokens=256)
print(ok)


query='what is RAG?' max_tokens=256


In [38]:
from pydantic import BaseModel, ValidationError

class AgentInput(BaseModel):
    query: str
    max_tokens: int

# invalid: max_tokens can't become an int -> Pydantic raises, loudly
try:
    AgentInput(query="hi", max_tokens="lots")
except ValidationError as e:
    print(e)


1 validation error for AgentInput
max_tokens
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='lots', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## 3. Build `AgentInput` and `AgentOutput`

This is the core deliverable. These two models define the **contract** for an agent: what goes in, what comes out. Every LangGraph node can trust the shape.

Notice each hint earns its place:
- `Optional[...]` — field may be absent / `None`
- `Literal[...]` — field restricted to known values
- `dict[str, int]` — structured sub-data (token usage)
- `Field(...)` — defaults, constraints, and docs in one place


In [39]:
from typing import Optional, Literal
from pydantic import BaseModel, Field


class AgentInput(BaseModel):
    """What the agent receives."""
    query: str = Field(..., min_length=1, description="User question")
    # Literal: model is one of a fixed, known set (typos rejected)
    model: Literal["claude-opus-4-8", "claude-haiku-4-5"] = "claude-haiku-4-5"
    max_tokens: int = Field(default=512, gt=0, le=4096)
    # Optional: a session may or may not exist
    session_id: Optional[str] = None


class AgentOutput(BaseModel):
    """What the agent returns."""
    # Literal: status is a closed set -> safe to branch on
    status: Literal["success", "error"]
    # Optional: no answer when status == "error"
    answer: Optional[str] = None
    # dict[str, int]: structured token usage, not a loose blob
    usage: dict[str, int] = Field(default_factory=dict)


inp = AgentInput(query="Explain cosine similarity")
print(inp)

out = AgentOutput(status="success", answer="It measures angle between vectors.",
                  usage={"prompt": 8, "completion": 14})
print(out)


query='Explain cosine similarity' model='claude-haiku-4-5' max_tokens=512 session_id=None
status='success' answer='It measures angle between vectors.' usage={'prompt': 8, 'completion': 14}


### Why a Pydantic model instead of a plain function / dict?

Same goal, two styles. Compare:


In [40]:
# --- Plain dict approach: no safety net ---
def build_input(query, model="claude-haiku-4-5", max_tokens=512):
    return {"query": query, "model": model, "max_tokens": max_tokens}

d = build_input("hi", model="gpt-4", max_tokens=-5)  # both wrong, no complaint
print("dict:", d)   # bad model + negative tokens ride along silently


dict: {'query': 'hi', 'model': 'gpt-4', 'max_tokens': -5}


In [41]:
from pydantic import ValidationError

# --- Pydantic approach: same fields, caught at creation ---
try:
    AgentInput(query="hi", model="gpt-4", max_tokens=-5)
except ValidationError as e:
    print(e)   # rejects unknown model AND max_tokens <= 0, with field names


2 validation errors for AgentInput
model
  Input should be 'claude-opus-4-8' or 'claude-haiku-4-5' [type=literal_error, input_value='gpt-4', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
max_tokens
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


**Takeaway:** the plain function trusts the caller. The Pydantic model *verifies* the caller. In an agent pipeline with many hops, verification at each boundary is what keeps failures local and readable.

## 4. `model_validate()` and `model_dump()` — the I/O methods

Real agent I/O is usually **dicts / JSON** (from an API request, an LLM's JSON output, a queue message). Two methods bridge dict ⇄ model:

| Method | Direction | Use |
|--------|-----------|-----|
| `Model.model_validate(data)` | **dict → model** | parse + validate incoming data |
| `instance.model_dump()` | **model → dict** | serialize for an API response / next node |
| `instance.model_dump_json()` | **model → JSON str** | send over the wire |


In [42]:
# Incoming request as a dict (e.g. JSON body) -> validate into a model
raw_request = {"query": "summarise this", "model": "claude-opus-4-8", "max_tokens": 1000}

inp = AgentInput.model_validate(raw_request)   # dict -> AgentInput (validated)
print("validated:", inp)
print("type:", type(inp).__name__)


validated: query='summarise this' model='claude-opus-4-8' max_tokens=1000 session_id=None
type: AgentInput


In [43]:
# Model -> dict, ready to return as an API response or pass to next node
out = AgentOutput(status="success", answer="Short summary.", usage={"prompt": 20, "completion": 5})

as_dict = out.model_dump()
print("dict :", as_dict)

as_json = out.model_dump_json(indent=2)
print("json :")
print(as_json)


dict : {'status': 'success', 'answer': 'Short summary.', 'usage': {'prompt': 20, 'completion': 5}}
json :
{
  "status": "success",
  "answer": "Short summary.",
  "usage": {
    "prompt": 20,
    "completion": 5
  }
}


### The full round-trip (the LangChain chain I/O pattern)

This is how Day-1 ties to real agent code: **validate the input → run the chain → validate/serialize the output.**


In [44]:
def run_chain(payload: dict) -> dict:
    # 1. dict in -> validate (bad input fails HERE, not deep in the chain)
    request = AgentInput.model_validate(payload)

    # 2. "the chain" — pretend LLM call using validated, trusted fields
    answer = f"[{request.model}] answered: {request.query}"

    # 3. build a validated output object
    result = AgentOutput(
        status="success",
        answer=answer,
        usage={"prompt": len(request.query.split()), "completion": 7},
    )

    # 4. model -> dict out (clean JSON-able response)
    return result.model_dump()


response = run_chain({"query": "what is LangGraph?", "model": "claude-opus-4-8"})
print(response)


{'status': 'success', 'answer': '[claude-opus-4-8] answered: what is LangGraph?', 'usage': {'prompt': 3, 'completion': 7}}


**`model_dump()` options worth knowing**
- `model_dump(exclude_none=True)` — drop `None` fields (smaller payloads)
- `model_dump(exclude={"usage"})` — hide internal fields from a public response
- `model_dump(mode="json")` — make values JSON-safe (e.g. `datetime` → string)


In [45]:
err = AgentOutput(status="error")   # answer + usage left at defaults

print("full      :", err.model_dump())
print("no None   :", err.model_dump(exclude_none=True))
print("hide usage:", err.model_dump(exclude={"usage"}))


full      : {'status': 'error', 'answer': None, 'usage': {}}
no None   : {'status': 'error', 'usage': {}}
hide usage: {'status': 'error', 'answer': None}


## 5. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **Type hints** | document in/out shape; editor + mypy catch misuse |
| **`Optional[str]`** | field may be missing (`answer` on error, `session_id`) — keeps "missing" distinct from "empty" |
| **`Literal[...]`** | lock `model` / `status` to a known set — typos rejected, safe to branch on |
| **`dict[str, int]`** | structured `usage` instead of a loose blob |
| **Pydantic `BaseModel`** | hints that **enforce at runtime** — bad data fails at the boundary |
| **`model_validate()`** | dict/JSON **in** → validated model |
| **`model_dump()`** | model **out** → dict/JSON for the next node or API |

### Done today
- [x] Type hints on every function
- [x] `AgentInput` + `AgentOutput` models
- [x] `model_validate()` (in) and `model_dump()` (out) round-trip

**Next — Day 02:** `TypedDict` for LangGraph State (and *why* TypedDict, not Pydantic, for state).

📚 Resources: Pydantic v2 docs · Real Python: type hints guide
